## Dadi Lab

In [1]:
two = 2
three = 3
two*three

6

In [32]:
cd ~/bioe-591-genomics/students/owenkanter/DadiLab/

/home/group/bioe-591-genomics/students/owenkanter/DadiLab


In [10]:
## Import Packages
import msprime
import dadi
import numpy as np

In [11]:
## Create Object
demography = msprime.Demography()

In [12]:
demography.add_population(name="pop", initial_size=500)  # present size

# ancestral size before T=800 generations ago was 10,000
demography.add_population_parameters_change(
    time=800, initial_size=10_000, population="pop"
)

PopulationParametersChange(time=800, initial_size=10000, growth_rate=None, population='pop')

In [14]:
demography

Demography(populations=[Population(initial_size=500, growth_rate=0, name='pop', description='', extra_metadata={}, default_sampling_time=None, initially_active=None, id=0)], events=[PopulationParametersChange(time=800, initial_size=10000, growth_rate=None, population='pop')], migration_matrix=array([[0.]]))

In [21]:
# Describe Simulation and samples themselves (changed random seed)

ts = msprime.sim_ancestry(
    samples={"pop": 20},
    demography=demography,
    sequence_length=5_000_000,
    recombination_rate=1e-8,
    random_seed=44,
)

In [22]:
# Adding Mutations (changed random seed)

mts = msprime.sim_mutations(ts, rate=1e-8, random_seed=45)

In [29]:
# Viewing and simulated genotypes

mts.genotype_matrix()

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 1, 1, ..., 1, 1, 1],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [1, 1, 1, ..., 1, 1, 1],
       [1, 1, 1, ..., 1, 1, 1]], shape=(2636, 40), dtype=int32)

In [30]:
# Saving simulated genotypes

with open("bottleneck_sim.vcf", "w") as f:
    mts.write_vcf(f)

In [60]:
# Read in VCF and pop file

vcf_file = "bottleneck_sim.vcf"
popfile = "pops.txt"
dd = dadi.Misc.make_data_dict_vcf(vcf_file, popfile)

In [61]:
# Summarize data as SFS

fs = dadi.Spectrum.from_data_dict(
    dd,
    pop_ids=["pop"],
    projections=[30],
    polarized=False,   # folded SFS
)
print("Spectrum sample size:", fs.sample_sizes)
print("Segregating sites:", fs.S())

Spectrum sample size: [30]
Segregating sites: 2513.0653337393405


In [62]:
# Specify Models

def snm(params, ns, pts): # define single population model, no free parameters
    xx = dadi.Numerics.default_grid(pts)
    phi = dadi.PhiManip.phi_1D(xx)
    fs_model = dadi.Spectrum.from_phi(phi, ns, (xx,))
    return fs_model

def two_epoch(params, ns, pts): # define bottlenneck model
    nu, T = params # two free parameters: scaled current pop size and time of split 
    xx = dadi.Numerics.default_grid(pts)
    phi = dadi.PhiManip.phi_1D(xx)
    phi = dadi.Integration.one_pop(phi, xx, T, nu)
    fs_model = dadi.Spectrum.from_phi(phi, ns, (xx,))
    return fs_model

In [63]:
# Calcilating and extrapolating phi

pts_l = [40, 50, 60]
snm_ex = dadi.Numerics.make_extrap_log_func(snm)
two_epoch_ex = dadi.Numerics.make_extrap_log_func(two_epoch)

In [64]:
# Fit models to data to compare results

model_snm = snm_ex([], fs.sample_sizes, pts_l)
theta_snm = dadi.Inference.optimal_sfs_scaling(model_snm, fs)
ll_snm = dadi.Inference.ll_multinom(model_snm, fs)

print("\nConstant-size model")
print("log-likelihood:", ll_snm)
print("theta:", theta_snm)


Constant-size model
log-likelihood: -523.2322780061425
theta: 636.0664450061497


In [65]:
# Estimating nu - setting bounds and parameters

p0 = [0.5, 0.1]  
lower_bound = [1e-3, 1e-4]
upper_bound = [20, 10]
p0_perturbed = dadi.Misc.perturb_params(
    p0, fold=1, lower_bound=lower_bound, upper_bound=upper_bound
)

In [66]:
# Likelihood Algorithm

popt = dadi.Inference.optimize_log(
    p0_perturbed,
    fs,
    two_epoch_ex,
    pts_l,
    lower_bound=lower_bound,
    upper_bound=upper_bound,
    verbose=1,
    maxiter=50,
)

1       , -488.322    , array([ 0.816625   ,  0.0751718  ])
2       , -488.495    , array([ 0.817442   ,  0.0751718  ])
3       , -488.301    , array([ 0.816625   ,  0.075247   ])
4       , -323.073    , array([ 0.299712   ,  0.0850967  ])
5       , -323.188    , array([ 0.300012   ,  0.0850967  ])
6       , -323.008    , array([ 0.299712   ,  0.0851818  ])
7       , -523.219    , array([ 0.0566043  ,  0.765973   ])
8       , -523.219    , array([ 0.0566609  ,  0.765973   ])
9       , -523.22     , array([ 0.0566043  ,  0.766739   ])
10      , -285.361    , array([ 0.232772   ,  0.11875    ])
11      , -285.419    , array([ 0.233005   ,  0.11875    ])
12      , -285.335    , array([ 0.232772   ,  0.118869   ])
13      , -272.539    , array([ 0.172748   ,  0.140722   ])
14      , -272.558    , array([ 0.172921   ,  0.140722   ])
15      , -272.54     , array([ 0.172748   ,  0.140863   ])
16      , -269.956    , array([ 0.144299   ,  0.131836   ])
17      , -269.966    , array([ 0.144443

In [67]:
# Fit model with optimized parameters

model_two = two_epoch_ex(popt, fs.sample_sizes, pts_l)
theta_two = dadi.Inference.optimal_sfs_scaling(model_two, fs)
ll_two = dadi.Inference.ll_multinom(model_two, fs)

print("\nTwo-epoch model")
print("best params [nu, T]:", popt)
print("log-likelihood:", ll_two)
print("theta:", theta_two)


Two-epoch model
best params [nu, T]: [0.03644814 0.08467168]
log-likelihood: -267.8853820745007
theta: 6130.073478479067


In [68]:
# AIC Scores

print("\nModel comparison")
print(f"Delta log-likelihood (two-epoch - constant): {ll_two - ll_snm:.3f}")

# constant model has k=0 free params in this formulation
# two_epoch has k=2
aic_snm = 2 * 0 - 2 * ll_snm
aic_two = 2 * 2 - 2 * ll_two

print(f"AIC constant: {aic_snm:.3f}")
print(f"AIC two-epoch: {aic_two:.3f}")


Model comparison
Delta log-likelihood (two-epoch - constant): 255.347
AIC constant: 1046.465
AIC two-epoch: 539.771


In [69]:
# Comparing true to inferred parameters

print("True parameters:")
print("nu =", 0.05)
print("T  =", 0.04)

print("\nInferred parameters:")
print("nu =", popt[0])
print("T  =", popt[1])

True parameters:
nu = 0.05
T  = 0.04

Inferred parameters:
nu = 0.03644813852310727
T  = 0.08467167713710745


In [70]:
# Scaling to real units

N_anc = 10_000  # known from simulation
nu_est, T_est = popt # label popt parameter estimates
N_curr_est = nu_est * N_anc # multiply Nu in dadi units by ancestral populaiton size to get current N_e
t_est = T_est * 2 * N_anc # multiply T in dadi units by 2N_e to get generations
print("Estimated current size:", N_curr_est)
print("Estimated bottleneck time (generations):", t_est)

Estimated current size: 364.4813852310727
Estimated bottleneck time (generations): 1693.433542742149


**Dadi Lab Interpretation**

*1. Interpreting AIC Scores*

Model comparison

Delta log-likelihood (two-epoch - constant): 255.347

AIC constant: 1046.465

AIC two-epoch: 539.771

The AIC value of the two-epoch model is lower indicating that it is the better performing model. This is also seen
by the large delta log-likelihood value.

*2. Parameter Values*

Estimated current size: 364.4813852310727

Estimated bottleneck time (generations): 1693.433542742149

True current size: 500

True bottleneck time (generations): 800

Dadi underestimated the current population size and overestimated the number of generations ago the bottleneck occurred.
The true current size seems reasonably close (a difference of 136 individuals); however, the estimated bottleneck time 
is two times greater than the true value.